# Experiment 8 — Association Rule Mining (Apriori) using Python

### Step 1: Load the dataset

In [ ]:
import pandas as pd

try:
    df = pd.read_csv('skill_profiles_long.csv')   # adjust the path if needed
    print("Dataset loaded successfully!")
    print("First few rows of the dataset:")
    print(df.head())
except FileNotFoundError:
    print("File not found. Please upload skill_profiles_long.csv to Colab.")

print("\nShape:", df.shape)
print("Distinct values per column:\n", df.nunique())
print("Missing values:\n", df.isnull().sum())
print("Exact duplicate rows:", df.duplicated().sum())

### Step 2: Clean the item labels and remove duplicates

In [ ]:
print("Distinct skill labels before:", df['skill'].nunique())
df['skill'] = df['skill'].str.strip().str.lower()
print("Distinct skill labels after: ", df['skill'].nunique())

before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows -> {len(df)} rows remain")

### Step 3: Group records into transactions

In [ ]:
transactions = df.groupby(['student_id', 'snapshot_date'])['skill'].apply(list).reset_index(name='items')
transaction_list = transactions['items'].tolist()

sizes = transactions['items'].apply(len)
print("Number of transactions:", len(transactions))
print("Average skills per transaction:", round(sizes.mean(), 2), "| min:", sizes.min(), "| max:", sizes.max())
print("Transactions per student (mean):", round(transactions.groupby('student_id').size().mean(), 2))
print(transactions.head())

### Step 4: Convert transactions to a Boolean matrix

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaction_list).transform(transaction_list)
encoded_df = pd.DataFrame(te_ary, columns=te.columns_)

print("First few rows of the encoded DataFrame:")
print(encoded_df.head())
print(f"\nTotal number of transactions: {len(encoded_df)}")
print(f"Number of unique items: {len(encoded_df.columns)}")

### Step 5 (Extension): Look at item frequencies to choose min_support

In [ ]:
import matplotlib.pyplot as plt

item_support = encoded_df.mean().sort_values(ascending=False)
print(item_support.round(3))

plt.figure(figsize=(9, 5))
item_support.plot(kind='bar')
plt.ylabel('Support'); plt.title('Support of each skill'); plt.tight_layout(); plt.show()

### Step 6: Find frequent itemsets with Apriori

In [ ]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(encoded_df, min_support=0.05, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

print("Number of frequent itemsets found:", len(frequent_itemsets))
print(frequent_itemsets['length'].value_counts().sort_index())
print(frequent_itemsets.sort_values(['length', 'support'], ascending=False).head())

### Step 7: Generate association rules

In [ ]:
from mlxtend.frequent_patterns import association_rules

try:
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
except TypeError:   # newer mlxtend versions also need the number of transactions
    rules = association_rules(frequent_itemsets, num_itemsets=len(encoded_df),
                              metric="confidence", min_threshold=0.5)

print("Number of rules with confidence >= 0.5:", len(rules))
rules = rules[rules['lift'] >= 1]
print("Number of rules after keeping lift >= 1:", len(rules))

### Step 8: Rank and interpret the rules

In [ ]:
def pretty(r):
    r = r.copy()
    r['antecedents'] = r['antecedents'].apply(lambda s: ', '.join(sorted(s)))
    r['consequents'] = r['consequents'].apply(lambda s: ', '.join(sorted(s)))
    return r

cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']

print("Top 10 rules by lift:")
print(pretty(rules.sort_values(['lift', 'confidence'], ascending=False))[cols].head(10).round(3))

print("\nTop 10 rules by confidence:")
print(pretty(rules.sort_values(['confidence', 'lift'], ascending=False))[cols].head(10).round(3))

### Step 9 (Extension): How do the thresholds change the results?

In [ ]:
for ms in (0.10, 0.05, 0.03):
    fi = apriori(encoded_df, min_support=ms, use_colnames=True)
    row = []
    for mc in (0.3, 0.5, 0.7, 0.9):
        try:
            r = association_rules(fi, metric="confidence", min_threshold=mc)
        except TypeError:
            r = association_rules(fi, num_itemsets=len(encoded_df), metric="confidence", min_threshold=mc)
        row.append(int((r['lift'] >= 1).sum()))
    print(f"min_support={ms}: itemsets={len(fi)}, rules (lift>=1) at confidence 0.3/0.5/0.7/0.9 -> {row}")

### Step 10 (Extension): Visualise the rules

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(rules['support'], rules['confidence'], c=rules['lift'], cmap='viridis', alpha=0.8)
plt.colorbar(sc, label='Lift')
ax.set_xlabel('Support'); ax.set_ylabel('Confidence'); ax.set_title('Association rules')
plt.show()

### Step 11: Export the basket matrix for Weka (link to Experiment 9)

In [ ]:
weka_df = encoded_df.apply(lambda col: col.map({True: 't', False: ''}))
weka_df.insert(0, 'transaction_id', transactions['student_id'] + '_' + transactions['snapshot_date'])
weka_df.to_csv('skill_baskets_weka.csv', index=False)
print(weka_df.shape)